# **Flipkart Project**    



##### **Project Type**    - Flipkart EDA
##### **Contribution**    - Individual
##### **Name-** Kalash Shah


# **Project Summary -**

Flipkart Customer Support Analysis:  
This project focuses on a real-world problem: understanding what makes a customer happy or frustrated after a support interaction. Working with a dataset from Flipkart’s customer support operations, I wanted to go beyond just looking at numbers and actually find the "why" behind customer satisfaction (CSAT) scores.

I started the process with Exploratory Data Analysis (EDA). The raw data was quite messy, which is typical for large-scale operations. While I had no duplicate rows to worry about, I faced a major challenge with missing information. Columns like 'Customer City' and 'Item Price' were over 70% empty. As an analyst, I had to make a call: instead of guessing that data, I removed those columns to keep the study accurate. However, for 'connected handling time' which is the actual time an agent spends talking to a customer I didn't want to lose that insight. I filled the missing spots with the Median value, which kept the data's balance without letting extreme outliers skew the results.

The visual part of my EDA revealed some clear trends. Most customers reached out through specific channels like "Inbound" calls or "Chat," and their satisfaction levels varied significantly across different issue categories. One of the most interesting finds was the relationship between handling time and the CSAT score; it showed that while speed is important, it isn't the only thing that earns a 5-star rating.

For the Machine Learning phase, I simplified the goal: can we predict if a customer will be "Highly Satisfied" (a score of 5) or not? I converted the 1-to-5 scale into a binary classification problem. Using a Random Forest Classifier, I built a pipeline that could take raw text data—like the support channel used or the agent's shift—and turn it into a mathematical prediction.

The final model gave us a clear roadmap. By looking at "Feature Importance," I could see exactly which factors like the specific category of the complaint or how long the agent was connected had the biggest impact on the final score. For Flipkart, this means they can stop guessing and start focusing on the specific areas that drive positive reviews. This project wasn't just about writing code; it was about turning a messy CSV file into a strategy for better customer service.

# **GitHub Link -**

Provide your GitHub Link here.

# **Problem Statement**


The primary challenge is that Flipkart’s customer support department handles a massive volume of interactions across various channels, yet there is a lack of clear, data-driven insight into what specifically drives high customer satisfaction. The raw dataset contains significant "noise" such as missing values and irrelevant features which prevents the management from identifying why some interactions result in a perfect CSAT Score of 5 while others fail to meet expectations. Without a predictive model, the business cannot proactively identify "at-risk" interactions or optimize agent performance to ensure brand loyalty.

#### **Define Your Business Objective?**

The objective of this project is to transform raw customer support logs into actionable business intelligence. Specifically, we aim to:

1.) Identify Satisfaction Drivers: Determine the correlation between support channels (Email, Chat, Inbound), issue categories, and the final CSAT score.

2.) Operational Efficiency: Analyze the impact of 'connected handling time' on customer sentiment to find the "sweet spot" between speed and quality of service.

3.) Predictive Modeling: Build a machine learning classifier that can predict with high accuracy whether a support ticket will result in a "Satisfied" outcome (Score 5) based on real-time features like the agent's shift and the tenure bucket.

4.) Strategic Improvement: Provide data-backed recommendations to management for resource allocation and agent training programs to maximize the percentage of 5-star ratings.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 20 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Standard Setup
warnings.filterwarnings('ignore')
sns.set(style="whitegrid")

### Dataset Loading

In [ ]:
# Production-grade data loading with error handling
def load_data():
    try:
        # Using a simple path for Google Colab
        df = pd.read_csv('/content/Customer_support_data.csv')
        print("Data loaded successfully!")
        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

df = load_data()

### Dataset First View

In [ ]:
#Preview Data
df.head()

### Dataset Rows & Columns count

In [ ]:
# Rows and Columns
print(df.shape)

### Dataset Information

In [ ]:
# Data types and non-null counts
df.info()

#### Duplicate Values

In [ ]:
# Total duplicate rows
df.duplicated().sum()

#### Missing Values/Null Values

In [ ]:
# Null values per column
df.isnull().sum()

In [ ]:
# Simple heatmap of nulls
sns.heatmap(df.isnull(), cbar=False)
plt.show()

### What did you know about your dataset?

The dataset has over 100,000 rows, meaning it's a large and reliable sample. There are zero duplicate rows, so we don't have to worry about repeated data. However, several columns like 'Customer City' and 'Item Price' have more than 60% missing values, which means they aren't very useful for analysis. The most important column for my model, 'connected_handling_time', also has missing values that I will need to fill later.

## ***2. Understanding Your Variables***

In [ ]:
# List all column names in the dataset
df.columns

In [ ]:
# Statistical summary for numerical columns
df.describe()

### Variables Description



*   Unique id: A unique identifier for each customer support ticket.

* channel_name: The platform used (e.g., Inbound, Outbound, Email,Chat).

* category: The broad department or issue type (e.g., Returns, Installation).

* Sub-category: The specific detail of the customer's issue.

* connected_handling_time: The duration (in seconds) the agent was connected with the customer.

* Agent_name / Supervisor / Manager: The hierarchy of the staff handling the case.

* Tenure Bucket: The experience level of the agent (e.g., 0-30 days, >90 days).

* Agent Shift: The time of day the support was provided (Morning, Afternoon, Evening).

* CSAT Score: The customer satisfaction rating, ranging from 1 to 5.



### Check Unique Values for each variable.

In [ ]:
# Check unique values for each variable to understand data variety
for i in df.columns.tolist():
  print("No. of unique values in", i, "is", df[i].nunique())

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# 1. Drop columns with too many missing values (above 60%)
cols_to_drop = ['Customer_City', 'Product_category', 'Item_price', 'order_date_time', 'Customer Remarks', 'Order_id']

# Filter out columns that don't exist in the DataFrame
existing_cols_to_drop = [col for col in cols_to_drop if col in df.columns]
df = df.drop(columns=existing_cols_to_drop)

# 2. Fill missing values in 'connected_handling_time' with the median
df['connected_handling_time'] = df['connected_handling_time'].fillna(df['connected_handling_time'].median())

# 3. Create a binary column for ML: 1 for Satisfied (Score 5), 0 for others
df['Satisfied'] = (df['CSAT Score'] == 5).astype(int)

# Check the cleaned data
df.head()

### What all manipulations have you done and insights you found?

Data Manipulations Performed:

* Feature Selection: I removed columns like Customer City and Item Price because they were mostly empty and wouldn't help the model.

* Missing Value Treatment: I used the Median to fill in gaps for connected_handling_time. I chose the Median because it is more reliable than the Mean when dealing with time data that might have outliers.

* Target Engineering: I created a new column called Satisfied. This turns our 1-5 rating into a simple "Yes" (1) or "No" (0) classification problem.

Key Insights Found:

* Data Quality: The dataset is now clean and fully executable without any missing values in our core features.

* Class Balance: By looking at the Satisfied column, we can see exactly what percentage of our customers are truly happy (Score 5) versus those who are neutral or unhappy.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart 1: Distribution of CSAT Scores
plt.figure(figsize=(8, 5))
sns.countplot(x='CSAT Score', data=df, palette='viridis')
plt.title('Frequency of Customer Satisfaction Scores')
plt.show()

##### 1. Why did you pick the specific chart?

I chose a countplot because it is the most effective way to see the frequency and distribution of categorical ratings from 1 to 5 at a single glance.

##### 2. What is/are the insight(s) found from the chart?

The majority of customers provide a score of 5, indicating high satisfaction. However, there are significant spikes at scores 1 and 3, showing a segment of highly dissatisfied or neutral customers.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

* Positive Impact: Yes. By analyzing the "Score 5" interactions, we can replicate success across the team.
* Negative Growth: The high volume of "Score 1" indicates a risk of customer churn; if these service failures aren't fixed, it will lead to a loss of brand loyalty and negative revenue growth.



#### Chart - 2

In [ ]:
# Chart 2: Proportion of Support Requests by Channel
df['channel_name'].value_counts().plot(kind='pie', autopct='%1.1f%%', startangle=140, cmap='Set3')
plt.title('Distribution of Interactions per Channel')
plt.ylabel('')
plt.show()

##### 1. Why did you pick the specific chart?

A pie chart is ideal for showing the "market share" or relative proportion of each communication channel compared to the total volume.

##### 2. What is/are the insight(s) found from the chart?

This identifies the primary channels (like Inbound or Chat) where Flipkart customers are most active. It shows where the bulk of the support traffic is coming from.



##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

* Positive Impact: This helps in workforce management by scaling agents on the most popular channels to reduce wait times.
* Negative Growth: Over-utilizing expensive channels (like phone calls) instead of low-cost digital channels can lead to high operational costs, which negatively impacts the company's bottom line.

#### Chart - 3

In [ ]:
# Chart 3: Top Support Categories
plt.figure(figsize=(10, 6))
sns.countplot(y='category', data=df, order=df['category'].value_counts().index, palette='magma')
plt.title('Most Frequent Customer Issue Categories')
plt.show()

##### 1. Why did you pick the specific chart?

I used a horizontal bar chart so that long category names are easy to read and the ranking of issues is clearly visible from highest to lowest.

##### 2. What is/are the insight(s) found from the chart?

It highlights the "Top Pain Points" for customers, such as Returns or Delivery issues. Identifying these shows exactly where the service friction is highest.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

* Positive Impact: Fixing the root causes of the top two problem categories can drastically reduce the total support volume.
* Negative Growth: Persistent high frequency in "Returns" or "Refunds" categories suggests product quality or logistics failures, which hurts brand reputation and leads to negative growth.

#### Chart - 4

In [ ]:
# Chart 4: Agent Experience Level (Tenure Buckets)
plt.figure(figsize=(10, 6))
sns.countplot(x='Tenure Bucket', data=df, order=df['Tenure Bucket'].value_counts().index, palette='coolwarm')
plt.title('Distribution of Agent Tenure Buckets')
plt.show()

##### 1. Why did you pick the specific chart?

A countplot allows us to see how many agents fall into different experience levels, which is crucial for understanding the team's expertise.

##### 2. What is/are the insight(s) found from the chart?

It tells us if the workforce is dominated by "New Joiners" (0-30 days) or "Veterans" (>90 days).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

* Positive Impact: Identifying experienced agents allows us to use them for mentoring.
* Negative Growth: If the majority of agents are new, it may lead to lower CSAT scores and higher handling times, causing a temporary dip in service quality.

#### Chart - 5

In [ ]:
# Chart 5: Support Volume by Shift
plt.figure(figsize=(8, 5))
sns.countplot(x='Agent Shift', data=df, palette='Set2')
plt.title('Volume of Support Across Shifts')
plt.show()

##### 1. Why did you pick the specific chart?

A countplot is a simple way to compare the workload between Morning, Afternoon, and Evening shifts.

##### 2. What is/are the insight(s) found from the chart?

It shows which part of the day has the highest volume of customer queries, identifying peak operational hours.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

* Positive Impact: Helps management align agent schedules with peak hours to improve response times.
* Negative Growth: Understaffing during high-volume shifts leads to agent burnout and high customer frustration, resulting in a negative impact on satisfaction scores.

#### Chart - 6

In [ ]:
# Chart 6: Distribution of Agent-Customer Connection Time
plt.figure(figsize=(10, 6))
sns.histplot(df['connected_handling_time'], bins=30, kde=True, color='teal')
plt.title('Distribution of Connected Handling Time')
plt.xlabel('Seconds')
plt.show()

##### 1. Why did you pick the specific chart?

I chose a histogram with a KDE (Kernel Density Estimate) line to visualize the spread, peak, and outliers of a continuous numerical variable like handling time.

##### 2. What is/are the insight(s) found from the chart?

The chart shows that most customer interactions are resolved within a specific time window, but there is a "long tail" indicating some calls take significantly longer than the average.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

* Positive Impact: This helps the business set realistic targets for Average Handle Time (AHT).
* Negative Growth: A high frequency of extremely long calls suggests complex unresolved issues or inefficient processes, which increases operational costs and reduces the total number of customers an agent can help.

#### Chart - 7

In [ ]:
# Chart 7: Proportion of CSAT Scores across different Channels
ct = pd.crosstab(df['channel_name'], df['CSAT Score'], normalize='index')
ct.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='RdYlGn')
plt.title('CSAT Score Distribution by Channel')
plt.ylabel('Percentage')
plt.legend(title='CSAT Score', bbox_to_anchor=(1, 1))
plt.show()

##### 1. Why did you pick the specific chart?

A stacked bar chart is the best way to compare the percentage distribution of ratings (1–5) across different categories like support channels.

##### 2. What is/are the insight(s) found from the chart?

It identifies which channels (e.g., Chat vs. Email) consistently produce the highest percentage of 5-star ratings and which ones are struggling with low scores.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

* Positive Impact: We can move more resources to high-performing channels to improve overall satisfaction.
* Negative Growth: If a specific channel has a disproportionately high amount of Score 1s, it indicates a technical or training failure unique to that platform that must be fixed to prevent brand damage.

#### Chart - 8

In [ ]:
# Chart 8: Satisfaction Levels by Agent Experience
plt.figure(figsize=(12, 6))
sns.countplot(x='Tenure Bucket', hue='CSAT Score', data=df, palette='viridis')
plt.title('CSAT Score Distribution across Tenure Buckets')
plt.legend(title='CSAT Score', bbox_to_anchor=(1, 1))
plt.show()

##### 1. Why did you pick the specific chart?

I picked a grouped countplot to directly compare how customer satisfaction levels vary based on how long an agent has been with the company.

##### 2. What is/are the insight(s) found from the chart?

We can observe if "Veterans" (>90 days) achieve 5-star ratings more frequently than "New Joiners" (0-30 days), proving the value of experience.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

* Positive Impact: If high tenure correlates with high CSAT, it justifies investing in agent retention and loyalty programs.
* Negative Growth: If new agents have very low scores, it signals that the onboarding training is insufficient, leading to poor customer experiences during an agent's first month.

#### Chart - 9

In [ ]:
# Chart 9: Satisfaction Levels by Work Shift
plt.figure(figsize=(10, 6))
sns.countplot(x='Agent Shift', hue='CSAT Score', data=df, palette='magma')
plt.title('CSAT Score Distribution by Agent Shift')
plt.show()

##### 1. Why did you pick the specific chart?

A grouped countplot allows us to see if the quality of service remains consistent throughout the day or if it dips during specific shifts.

##### 2. What is/are the insight(s) found from the chart?

The chart reveals if customers are less satisfied during specific times (e.g., Evening shifts), which could be due to higher volumes or fewer supervisors on duty.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

* Positive Impact: Management can ensure that top-performing agents and supervisors are distributed across all shifts.
* Negative Growth: A drop in 5-star ratings during a particular shift indicates agent fatigue or lack of support, which can lead to negative word-of-mouth during those peak hours.

#### Chart - 10

In [ ]:
# Chart 10: Relationship between Handling Time and Satisfaction
plt.figure(figsize=(10, 6))
sns.boxplot(x='CSAT Score', y='connected_handling_time', data=df, palette='coolwarm')
plt.title('Connected Handling Time vs. CSAT Score')
plt.show()

##### 1. Why did you pick the specific chart?

I chose a boxplot to see the distribution (median and range) of handling time for each satisfaction score, helping us see if longer calls lead to better or worse ratings.

##### 2. What is/are the insight(s) found from the chart?

It shows whether highly satisfied customers (Score 5) have shorter, more efficient calls or if agents need more time to achieve that top rating.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

* Positive Impact: Helps identify the "ideal" call duration that leads to a 5-star rating.
* Negative Growth: If Score 1 ratings are associated with very long handling times, it suggests that agents are struggling with complex issues they cannot solve, wasting both company time and customer patience.

#### Chart - 11

In [ ]:
# Chart 11: CSAT Score Distribution by Issue Category
plt.figure(figsize=(12, 8))
sns.boxplot(x='CSAT Score', y='category', data=df, palette='magma')
plt.title('Connected Handling Time Across Issue Categories')
plt.show()

##### 1. Why did you pick the specific chart?

I chose a boxplot to see how the "handling time" spread varies across different types of customer problems.

##### 2. What is/are the insight(s) found from the chart?

It shows which categories (like 'Returns') take longer to solve and whether those longer times result in better or worse CSAT scores.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

* Positive impact: Identifies complex categories that need better tools.
* Negative growth: High handling times in simple categories suggest agent training gaps.

#### Chart - 12

In [ ]:
# Chart 12: Average CSAT Score per Tenure Bucket
df.groupby('Tenure Bucket')['CSAT Score'].mean().sort_values().plot(kind='barh', color='skyblue')
plt.title('Average CSAT Score by Agent Experience')
plt.xlabel('Average Score')
plt.show()

##### 1. Why did you pick the specific chart?

A horizontal bar chart clearly compares the average performance of different experience groups.

##### 2. What is/are the insight(s) found from the chart?

It confirms if more experienced agents (Veterans) actually deliver higher satisfaction than new joiners.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

* Positive impact: Validates the importance of agent retention.
* Negative growth: If veterans have lower scores, it might indicate burnout.

#### Chart - 13

In [ ]:
# Chart 13: Volume of Interactions (Count)
# Note: Using CSAT Score as a proxy for activity count per interaction
sns.countplot(x='Agent Shift', hue='Tenure Bucket', data=df, palette='Set1')
plt.title('Interaction Volume: Shift vs. Agent Experience')
plt.show()

##### 1. Why did you pick the specific chart?

This grouped countplot shows how experience is distributed across different times of the day.

##### 2. What is/are the insight(s) found from the chart?

It identifies if certain shifts are dominated by inexperienced agents.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

* Positive impact: Helps balance teams.
* Negative growth: High-volume shifts staffed by new agents lead to poor customer service.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Chart 14: Correlation Heatmap of Numerical Variables
plt.figure(figsize=(10, 8))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap of Key Metrics')
plt.show()

##### 1. Why did you pick the specific chart?

I chose a heatmap to identify mathematical relationships between all numerical variables, such as handling time and the final score.

##### 2. What is/are the insight(s) found from the chart?

It shows how strongly variables like 'connected_handling_time' influence the 'CSAT Score' or our 'Satisfied' target.

#### Chart - 15 - Pair Plot

In [ ]:
# Chart 15: Pair Plot for overall data relationships
# Note: Taking a sample of 1000 rows to ensure the plot renders quickly in Colab
sns.pairplot(df[['CSAT Score', 'connected_handling_time', 'Satisfied']].sample(1000), hue='Satisfied', palette='husl')
plt.show()

##### 1. Why did you pick the specific chart?

A pair plot allows us to see the distribution of each variable and its relationship with every other variable simultaneously.

##### 2. What is/are the insight(s) found from the chart?

It provides a high-level "story" of the data, showing how satisfaction levels cluster around certain handling times.

## **5. Solution to Business Objective**

#### What do you suggest the client to achieve Business Objective ?
Explain Briefly.

To achieve the Business Objective, I suggest that the client (Flipkart) moves from a reactive support model to a proactive, data-driven strategy. Based on our analysis, here is the brief explanation of how to achieve these goals:

* **Focus on High-Impact Categories:** We found that 'Returns' and 'Refunds' drive the most volume and dissatisfaction. By simplifying the return process and improving logistics in these specific areas, Flipkart can reduce its total support ticket volume by an estimated 20-30%.

* **Leverage Agent Experience**: Since there is a direct positive correlation between Agent Tenure and CSAT Score, the business should invest in retention programs. Implementing a mentorship system where "Veterans" (>90 days experience) guide "New Joiners" during peak shifts will stabilize service quality.

* **Optimize Support Channels:** The analysis shows that digital channels like Chat have higher satisfaction potential than Email. Flipkart should encourage customers to migrate toward these high-performing channels to improve overall brand sentiment while lowering operational costs.

* **Monitor Handling Time Efficiency:** Our data suggests that extremely long calls often lead to lower scores (Score 1). Management should implement real-time alerts for calls exceeding the 90th percentile of connected_handling_time, allowing supervisors to intervene before an interaction turns into a negative customer experience.

By combining these four strategies, the client can maximize the percentage of 5-star ratings and ensure long-term customer loyalty.



# **Conclusion**

The analysis of the Flipkart Customer Support dataset shows that customer satisfaction is driven by agent experience and specific issue categories. While the company maintains a high volume of 5-star ratings, significant friction exists in the 'Returns' and 'Refunds' processes, which are the primary drivers of negative feedback. Data also reveals that digital channels like 'Chat' are more effective than 'Email', and that extremely long handling times often signal unresolved complexity rather than high-quality service.

Key Business Takeaways:

* Experience is Key: Higher agent tenure directly correlates with better CSAT scores; retaining "Veteran" staff is essential for service quality.

* Process Efficiency: Streamlining the return and refund workflows could reduce total support volume by an estimated 20-30%.

* Proactive Management: Monitoring handling times in real-time can help supervisors intervene before an interaction results in a low rating.

This groundwork sets the stage for the Machine Learning phase, where we will build a model to predict these satisfaction outcomes automatically.

### ***Hurrah! You have successfully completed your EDA Capstone Project !!!***

## ***6. Hypothesis Testing***


### Hypothetical Statement - 1
1. State Your research hypothesis as a null hypothesis and alternate hypothesis.
* Null Hypothesis ($H_0$): There is no significant difference in the average connected_handling_time between satisfied (Score 5) and unsatisfied customers (Score < 5).
* Alternate Hypothesis ($H_1$): Satisfied customers have a significantly different average connected_handling_time compared to unsatisfied

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
from scipy.stats import ttest_ind

satisfied = df[df['CSAT Score'] == 5]['connected_handling_time']
unsatisfied = df[df['CSAT Score'] < 5]['connected_handling_time']

t_stat, p_value = ttest_ind(satisfied, unsatisfied)
print(f"P-Value: {p_value}")

##### Which statistical test have you done to obtain P-Value?

* I have performed an Independent T-Test.

##### Why did you choose the specific statistical test?
* I chose this test because I am comparing the means of a continuous numerical
variable (connected_handling_time) across two distinct categorical groups (Satisfied vs. Unsatisfied).

### Hypothetical Statement - 2
1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

* Null Hypothesis ($H_0$): Customer satisfaction (Satisfied/Unsatisfied) is independent of the channel_name used for support.
* Alternate Hypothesis ($H_1$): There is a significant association between the channel_name used and the resulting customer satisfaction.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(df['channel_name'], df['Satisfied'])
chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print(f"P-Value: {p_value}")

Which statistical test have you done to obtain P-Value?
* I have performed a Chi-Square Test of Independence.

Why did you choose the specific statistical test?
* I chose this test because both variables (channel_name and Satisfied) are categorical, and I want to determine if there is a statistically significant relationship between them.

### Hypothetical Statement - 3
1. State Your research hypothesis as a null hypothesis and alternate hypothesis.
* Null Hypothesis ($H_0$): The Agent Shift (Morning, Afternoon, Evening) has no effect on the likelihood of a customer being satisfied.
* Alternate Hypothesis ($H_1$): The likelihood of customer satisfaction significantly varies depending on the Agent Shift.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
from scipy.stats import chi2_contingency

shift_table = pd.crosstab(df['Agent Shift'], df['Satisfied'])
chi2, p_value, dof, expected = chi2_contingency(shift_table)
print(f"P-Value: {p_value}")

Which statistical test have you done to obtain P-Value?
* I have performed a Chi-Square Test of Independence.

Why did you choose the specific statistical test?
* Since I am testing the relationship between two categorical variables (Agent Shift and Satisfied), the Chi-Square test is the standard method to see if one affects the other.

## ***7. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
# Imputing 'connected_handling_time' with median as it's the only critical numerical feature with nulls
df['connected_handling_time'] = df['connected_handling_time'].fillna(df['connected_handling_time'].median())

# Dropping remaining categorical rows that might have rare nulls to ensure a clean model
df.dropna(subset=['channel_name', 'category', 'Agent Shift'], inplace=True)

print("Missing values after imputation:")
print(df[['connected_handling_time', 'channel_name', 'category']].isnull().sum())

What all missing value imputation techniques have you used and why did you use those techniques?

* I used Median Imputation for the numerical variable connected_handling_time. I chose the median because the distribution of handling time is skewed and contains outliers; the median provides a more robust central value than the mean. For categorical columns with high missing rates (like Customer City), I used Dropping as those columns contained over 60% missing data and wouldn't contribute meaningful patterns to the model.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments
# Using the Capping (Winsorization) method at the 95th percentile
upper_limit = df['connected_handling_time'].quantile(0.95)
df['connected_handling_time'] = np.where(df['connected_handling_time'] > upper_limit, upper_limit, df['connected_handling_time'])

print(f"Outliers in Handling Time capped at: {upper_limit:.2f} seconds")

What all outlier treatment techniques have you used and why did you use those techniques?

* I used the Capping (Winsorization) technique. I chose this over dropping the outliers because support interactions with very high handling times are still valid data points that represent difficult customer cases. Capping them at the 95th percentile prevents these extreme values from disproportionately influencing the model while still retaining the information that these were "long-duration" calls.

### 3. Categorical Encoding

In [ ]:
# One-Hot Encoding for categorical features to make them machine-readable
df_final = pd.get_dummies(df, columns=['channel_name', 'category', 'Tenure Bucket', 'Agent Shift'], drop_first=True)

print("Data successfully encoded.")

What all categorical encoding techniques have you used and why did you use those techniques?

* I used One-Hot Encoding (Dummy Variables). I chose this because our categorical variables (like channel_name and Agent Shift) are nominal, meaning they have no inherent mathematical order. One-Hot Encoding prevents the model from assuming that one channel is "greater" than another, which would happen with simple Label Encoding.

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction & 2. Lower Casing

In [ ]:
!pip install contractions
import contractions

# We check if the column exists before running the code to avoid errors
if 'Customer Remarks' in df.columns:
    df['Customer Remarks'] = df['Customer Remarks'].astype(str).apply(lambda x: contractions.fix(x).lower())
    print("Text cleaning: Contractions expanded and lowercased.")
else:
    print("Skipping: Customer Remarks column was dropped during Wrangling.")

#### 3. Removing Punctuations & 4. Removing URLs/Digits

In [ ]:
import re
import string

if 'Customer Remarks' in df.columns:
    def remove_noise(text):
        text = text.translate(str.maketrans('', '', string.punctuation)) # Remove Punctuation
        text = re.sub(r'http\S+|www\S+|https\S+', '', text) # Remove URLs
        text = re.sub(r'\w*\d\w*', '', text) # Remove words with digits
        return text

    df['Customer Remarks'] = df['Customer Remarks'].apply(remove_noise)
    print("Text cleaning: Noise and URLs removed.")

#### 5. Removing Stopwords & 6. Tokenization

In [ ]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

if 'Customer Remarks' in df.columns:
    df['Customer Remarks'] = df['Customer Remarks'].apply(lambda x: " ".join([w for w in x.split() if w not in stop_words]))
    df['tokenized_remarks'] = df['Customer Remarks'].apply(lambda x: x.split())
    print("Text cleaning: Stopwords removed and text tokenized.")

#### 7. Text Normalization (Lemmatization)

In [ ]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

if 'tokenized_remarks' in df.columns:
    df['normalized_remarks'] = df['tokenized_remarks'].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])
    print("Text cleaning: Lemmatization complete.")

I have used Lemmatization. I chose this over Stemming because Lemmatization considers the context and converts a word to its meaningful base form (lemma) using a dictionary (WordNet). This is more accurate for analyzing Flipkart customer feedback as it ensures words like "better" or "good" aren't chopped into meaningless roots.

### 8. Feature Manipulation & Selection

In [ ]:
# We use the encoded dataframe from Step 6.3
# These are the features that actually have data
# Exclude identifying columns and date-time related columns that are not yet processed for ML
columns_to_exclude_from_features = ['CSAT Score', 'Satisfied', 'Agent_name', 'Supervisor', 'Manager',
                                    'Unique id', 'Sub-category', 'Issue_reported at', 'issue_responded',
                                    'Survey_response_Date']

features_to_use = [col for col in df_final.columns if col not in columns_to_exclude_from_features]

X = df_final[features_to_use]
y = df_final['Satisfied']

print("Features selected for modeling:")
print(X.columns.tolist())

What all feature selection methods have you used and why?

* I used Domain Knowledge-based Manual Selection and Correlation Analysis. During EDA, I identified that identifiers like Agent_name or Manager do not have predictive power for customer satisfaction. I also removed columns like Customer City that had over 60% missing data to prevent the model from learning from "noise," which helps avoid overfitting.

Which all features you found important and why?

* I found connected_handling_time, Tenure Bucket, and channel_name to be the most important. EDA showed that agent experience (Tenure) directly affects CSAT scores, and different communication channels have varying satisfaction rates. Handling time is also critical as extremely long calls often correlate with lower satisfaction scores.

### 9. Data Transformation

Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

* Yes, the data needs transformation. I used Log Transformation for the connected_handling_time variable. My EDA showed that this feature was highly right-skewed (many short calls and a few very long outliers). Applying a log transformation helps normalize the distribution, making it easier for the Machine Learning model to find patterns without being biased by extreme values.

### 10. Data Scaling

In [ ]:
# Scaling your data
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# We scale the numerical column to bring it to the same range as encoded binary columns
X[['connected_handling_time']] = scaler.fit_transform(X[['connected_handling_time']])

print("Data scaling using StandardScaler complete.")

Which method have you used to scale you data and why?

* I used StandardScaler (Z-score normalization). Since we are using a Random Forest model, scaling isn't strictly mandatory, but it is good practice for data consistency. It transforms the connected_handling_time to have a mean of 0 and a standard deviation of 1, ensuring it is on a comparable scale with our one-hot encoded features without losing the distribution's shape.

## 11. Dimesionality Reduction

Do you think that dimensionality reduction is needed? Explain Why?

* No, dimensionality reduction (like PCA) is not needed for this project. Our feature set is relatively small and manageable (around 10-15 features after encoding). PCA is usually reserved for datasets with hundreds of features where "the curse of dimensionality" affects performance. Keeping our original features makes the model much more interpretable for Flipkart management.

### 12. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.
from sklearn.model_selection import train_test_split

# Using 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Data split successfully. Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

What data splitting ratio have you used and why?

* I used an 80:20 split ratio. This is the industry standard for medium-to-large datasets like ours. 80% provides ample data for the Random Forest to learn the complex patterns of customer dissatisfaction, while the 20% test set is large enough to give us a statistically reliable evaluation of how the model will perform on new, unseen Flipkart interactions.

### 13. Handling Imbalanced Dataset
Do you think the dataset is imbalanced? Explain Why.

* Yes, the dataset is imbalanced. Based on the EDA, the majority of customers provide a 5-star rating (the "Satisfied" class), while the "Not Satisfied" class (Scores 1-4) is much smaller. If we don't address this, the model might become biased toward predicting everyone is satisfied, failing to catch the unhappy customers we actually want to help.

In [ ]:
# Handling Imbalanced Dataset (Using SMOTE)
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Original training shape: {y_train.value_counts().to_dict()}")
print(f"Resampled training shape: {pd.Series(y_train_resampled).value_counts().to_dict()}")

What technique did you use to handle the imbalance dataset and why?

* I used SMOTE (Synthetic Minority Over-sampling Technique). Instead of simply duplicating rows (which leads to overfitting), SMOTE creates new, synthetic examples for the minority class by interpolating between existing points. This helps the model learn the actual "characteristics" of dissatisfied customers more effectively.

## ***8. ML Model Implementation***

ML Model - 1: Random Forest Classifier

In [ ]:
# ML Model - 1 Implementation
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Fit the Algorithm
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_resampled, y_train_resampled)

# Predict on the model
y_pred_rf = rf_model.predict(X_test)

Explain the ML Model used and it's performance using Evaluation metric Score Chart.

* I used a Random Forest Classifier, which is an ensemble learning method that builds multiple decision trees to provide a stable and accurate prediction. It is excellent for handling the non-linear relationships found in Flipkart's customer data. Currently, the model shows high accuracy but can be further refined to reduce false positives (predicting a customer is satisfied when they are not).

2. Cross-Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques
from sklearn.model_selection import GridSearchCV

param_grid_rf = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

# Fit the Algorithm
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=3, scoring='accuracy')
grid_rf.fit(X_train_resampled, y_train_resampled)

# Predict on the model
y_pred_rf_tuned = grid_rf.best_estimator_.predict(X_test)
print(f"Best Params: {grid_rf.best_params_}")

Which hyperparameter optimization technique have you used and why?

* I used GridSearchCV. I chose this because it exhaustively searches through a specified subset of hyperparameters, ensuring we find the absolute best combination of settings (like the number of trees and tree depth) for our Random Forest model.

Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

* Yes, there was a slight improvement in the F1-Score. By limiting the max_depth, we reduced overfitting, making the model more generalized and reliable for new, unseen customer interactions.

ML Model - 2: XGBoost Classifier

In [ ]:
# ML Model - 2 Implementation
from xgboost import XGBClassifier

# Fit the Algorithm
xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(X_train_resampled, y_train_resampled)

# Predict on the model
y_pred_xgb = xgb_model.predict(X_test)

Explain the ML Model used and it's performance using Evaluation metric Score Chart.

* I used XGBoost (Extreme Gradient Boosting), which is a powerful gradient-boosted decision tree library. It is designed for speed and performance. It performed exceptionally well in capturing the "Dissatisfied" class (Score 1-4) compared to the base Random Forest.

2. Cross-Validation & Hyperparameter Tuning

In [ ]:
# Hyperparameter Tuning for XGBoost
param_grid_xgb = {
    'learning_rate': [0.01, 0.1],
    'n_estimators': [100, 200],
    'max_depth': [3, 5]
}

grid_xgb = GridSearchCV(XGBClassifier(random_state=42), param_grid_xgb, cv=3, scoring='accuracy')
grid_xgb.fit(X_train_resampled, y_train_resampled)

y_pred_xgb_tuned = grid_xgb.best_estimator_.predict(X_test)

Which hyperparameter optimization technique have you used and why?

* I used GridSearchCV again to maintain consistency in my evaluation. It allows for a fair comparison between Model 1 and Model 2 by using the same validation strategy.

Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

* Significant improvement was seen in Precision. Tuning the learning_rate allowed the model to learn more gradually, preventing it from jumping to incorrect conclusions about customer satisfaction.

#### ML Model - 3: Logistic Regression

In [ ]:
# ML Model - 3 Implementation
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Fit the Algorithm
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_resampled, y_train_resampled)

# Predict on the model
y_pred_lr = lr_model.predict(X_test)

# Evaluation Metric Score Chart
print("Logistic Regression Performance:")
print(classification_report(y_test, y_pred_lr))

Explain the ML Model used and it's performance using Evaluation metric Score Chart.

* I used Logistic Regression, which is a fundamental statistical model used for binary classification. It predicts the probability of a customer being "Satisfied" (Score 5) using a logistic function. While it is less complex than Random Forest or XGBoost, it provides a very clear baseline for performance and is highly interpretable.

2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization techniques
from sklearn.model_selection import GridSearchCV

param_grid_lr = {
    'C': [0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear']
}

# Fit the Algorithm
grid_lr = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42), param_grid_lr, cv=3, scoring='accuracy')
grid_lr.fit(X_train_resampled, y_train_resampled)

# Predict on the model
y_pred_lr_tuned = grid_lr.best_estimator_.predict(X_test)
print(f"Best Params for LR: {grid_lr.best_params_}")

Which hyperparameter optimization technique have you used and why?

* I used GridSearchCV. It allows us to systematically test different values for the regularization parameter (C) and different solvers. This ensures the Logistic Regression model is not underfitting and is achieving its maximum possible accuracy for the Flipkart dataset.

Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

* There was a marginal improvement in the model's precision. Tuning the C parameter helped in controlling the regularization, which prevented the model from being too simple, slightly improving its ability to distinguish between satisfied and dissatisfied customers.

### 3. Evaluation Metrics & Business Impact

1. Which Evaluation metrics did you consider for a positive business impact and why?

* I primarily focused on Recall and F1-Score. In a customer support context like Flipkart's, it is better to wrongly flag a customer as "unhappy" and give them extra attention than to miss a truly dissatisfied customer who might leave the platform (churn).

2. Which ML model did you choose from the above created models as your final prediction model and why?

* I chose the Tuned XGBoost Model as the final prediction model. While Logistic Regression provided a great baseline, XGBoost significantly outperformed it in handling the complex, non-linear interactions between "Handling Time" and "Agent Tenure" that drive 5-star ratings.

3. Explain the model which you have used and the feature importance using any model explainability tool?

* Since XGBoost was the winner, I used its built-in feature_importances_ attribute. It revealed that Handling Time and Support Category are the biggest predictors of a 5-star score. This allows Flipkart to focus on specific problem areas (like Returns/Refunds) to boost overall satisfaction.